 Question 1: Clean and Convert `total_sqft` Values

Convert `total_sqft` values into numeric format. For ranges such as `2100 - 2850`, calculate the average. Convert invalid or unsupported area values to `NaN`.

In [1]:
# Import pandas for data analysis
import pandas as pd

# Load the bengaluru house data csv file
df = pd.read_csv("Bengaluru_House_Data.csv")

# Display the first 5 rows
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [2]:
# Check the number of rows and columns
print(df.shape)

# Check column name, data types, and missing values
df.info()

(13320, 9)
<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 936.7 KB


In [3]:
# Display same unique values from the total_sqft column
df["total_sqft"].unique()[:30]

<StringArray>
[       '1056',        '2600',        '1440',        '1521',        '1200',
        '1170',        '2732',        '3300',        '1310',        '1020',
        '1800',        '2785',        '1000',        '1100',        '2250',
        '1175',        '1180',        '1540',        '2770',         '600',
        '1755',        '2800',        '1767',         '510',        '1250',
         '660',        '1610',        '1151',        '1025', '2100 - 2850']
Length: 30, dtype: str

In [4]:
# Display total_sqft values containing a hyphen
df[df["total_sqft"].str.contains("-", na=False)]["total_sqft"].head()

30     2100 - 2850
56     3010 - 3410
81     2957 - 3450
122    3067 - 8156
137    1042 - 1105
Name: total_sqft, dtype: str

In [5]:
import numpy as np

def convert_sqft(x):
    try:
        parts = x.split("-")
        if len(parts) == 2:
            return (float(parts[0].strip()) + float(parts[1].strip())) / 2
        return float(x)
    except:
        return np.nan

df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [6]:
df["total_sqft"].head()

0    1056.0
1    2600.0
2    1440.0
3    1521.0
4    1200.0
Name: total_sqft, dtype: float64

In [7]:
# Check a range value after cleaning
print(df.loc[30, "total_sqft"])

# Check the number of missing values
print(df["total_sqft"].isna().sum())

2475.0
46


In [8]:
# Verify the current dataset after cleaning total_sqft
print(df.shape)
print(df["total_sqft"].dtype)
print(df["total_sqft"].isna().sum())

(13320, 9)
float64
46


Question 2: Extract BHK from the Size Column

Extract the numerical value from the `size` column (e.g., `2 BHK`, `4 Bedroom`) and store it as an integer in a new column named `bhk`. Drop rows where the BHK value cannot be determined.

In [9]:
# Question 2: Inspect the size column
df["size"].unique()[:20]

<StringArray>
[    '2 BHK', '4 Bedroom',     '3 BHK',     '4 BHK', '6 Bedroom', '3 Bedroom',
     '1 BHK',      '1 RK', '1 Bedroom', '8 Bedroom', '2 Bedroom', '7 Bedroom',
     '5 BHK',     '7 BHK',     '6 BHK', '5 Bedroom',    '11 BHK',     '9 BHK',
         nan, '9 Bedroom']
Length: 20, dtype: str

In [10]:
# Extract BHK number from size
df["bhk"] = df["size"].str.extract(r"(\d+)").astype("Int64")

In [11]:
# Verify the new column
df[["size", "bhk"]].head()

,size,bhk
0,2 BHK,2
1,4 Bedroom,4
2,3 BHK,3
3,3 BHK,3
4,2 BHK,2


In [12]:
# Check missing BHK values
df["bhk"].isna().sum()

np.int64(16)

In [13]:
# Remove rows where BHK is missing
df = df.dropna(subset=["bhk"])

In [14]:
# Check the dataset after removing missing BHK values
print(df.shape)
print(df["bhk"].isna().sum())

(13304, 10)
0


Question 3: Filter Properties by Minimum Area per BHK

Remove properties where the total square footage divided by the number of BHK is less than 300 square feet.

In [15]:
# Filter properties with at least 300 sqft per BHK
df = df[(df["total_sqft"] / df["bhk"]) >= 300]

In [16]:
# Check the number of rows and columns after filtering
print(df.shape)

# Verify that every remaining property meets the minimum area rule
print((df["total_sqft"] / df["bhk"] < 300).sum())

(12514, 10)
0


Question 4: Identify and Remove Bathroom Anomalies

Identify properties where the number of bathrooms (`bath`) is greater than the number of BHK plus 2, and remove those rows from the core analytical dataset.

In [17]:
# Identify properties where bathrooms exceed BHK by more than 2
bath_anomalies = df[df["bath"] > df["bhk"] + 2]

# Display the number of anomalies and sample rows
print("Number of bathroom anomalies:", len(bath_anomalies))
bath_anomalies[["location", "size", "bhk", "bath"]].head()


Number of bathroom anomalies: 9


,location,size,bhk,bath
1078,BTM 1st Stage,9 Bedroom,9,14.0
2620,Sathya Sai Layout,6 BHK,6,9.0
6838,Rajaji Nagar,5 BHK,5,8.0
7709,Chikkabanavar,4 Bedroom,4,7.0
9974,Sector 1 HSR Layout,3 Bedroom,3,6.0


In [18]:
# Remove bathroom anomalies from the main dataset
df = df[df["bath"] <= df["bhk"] + 2]

In [19]:
# Verify the dataset after removing bathroom anomalies
print(df.shape)
print((df["bath"] > df["bhk"] + 2).sum())

(12448, 10)
0


Question 5: Identify the Top 5 Locations by Mean Price per Square Foot

Calculate `price_per_sqft` as `(price × 100,000) / total_sqft`. Group properties by location, exclude locations with fewer than 15 listings, and identify the top 5 locations with the highest mean `price_per_sqft`.

In [20]:
# Calculate price per square foot
df["price_per_sqft"] = (df["price"] * 100000) / df["total_sqft"]

In [21]:
# Display price per square foot for the first 5 properties
df[["location", "total_sqft", "price", "price_per_sqft"]].head()

,location,total_sqft,price,price_per_sqft
0,Electronic City Phase II,1056.0,39.07,3699.810606
1,Chikka Tirupathi,2600.0,120.00,4615.384615
2,Uttarahalli,1440.0,62.00,4305.555556
3,Lingadheeranahalli,1521.0,95.00,6245.890861
4,Kothanur,1200.0,51.00,4250.000000


In [22]:
# Calculate the mean price per square foot for each location
location_mean_price = df.groupby("location")["price_per_sqft"].mean()

In [23]:
# Display the first 5 location averages
location_mean_price.head()

location
Anekal                 4558.404558
Banaswadi              6641.366224
Basavangudi            7462.686567
Devarabeesana Halli    7900.452568
Devarachikkanahalli    4969.906245
Name: price_per_sqft, dtype: float64

In [24]:
# Count property listings in each location
location_counts = df["location"].value_counts()

# keep locations with at least 15 property listings
valid_location = location_counts[location_counts >= 15].index

In [25]:
# Keep only locations with at least 15 property listings
filtered_location_mean = location_mean_price[
    location_mean_price.index.isin(valid_location)
    ]

# Display the number of qualifying locations
print(filtered_location_mean.shape[0])

176


In [26]:
# Sort locations by mean price per square foot in descending order
top_5_locations = filtered_location_mean.sort_values(ascending=False).head(5)

# Display the top 5 locations
top_5_locations

location
Indira Nagar    13916.496675
Rajaji Nagar    13553.529547
Malleshwaram    13497.861130
Kodihalli       12745.704413
Benson Town     12688.815074
Name: price_per_sqft, dtype: float64

Question 6: The Executive Cross-Tabulation Matrix
Assignment requirement: Create a pivot table showing the median property price, with area_type as rows and the number of balconies as columns. Fill missing combinations with 0.

In [27]:
# Create a pivot table of median prices by area type and balcony count
median_price_pivot = pd.pivot_table(
    df,
    values="price",
    index="area_type",
    columns="balcony",
    aggfunc="median",
    fill_value=0
)

# Display the pivot table
median_price_pivot

balcony,0.0,1.0,2.0,3.0
area_type,,,,
Built-up Area,71.000,57.000,70.0,99.0
Carpet Area,47.405,60.000,70.0,91.0
Plot Area,80.000,126.500,200.0,250.0
Super built-up Area,56.725,56.375,68.0,82.0


Question 7 — Volatile Real Estate Ecosystems
Assignment requirement: Find the top 3 locations with the highest standard deviation of price_per_sqft, considering only locations with at least 10 listings

In [28]:
# Calculate the standard deviation of price per square foot for each location
location_price_std = df.groupby("location")["price_per_sqft"].std()

# Count property listings in each location
location_listing_counts = df["location"].value_counts()

# Keep locations with at least 10 listings
valid_locations_10 = location_listing_counts[location_listing_counts >= 10].index

# Filter standard deviations to qualifying locations
filtered_price_std = location_price_std[
    location_price_std.index.isin(valid_locations_10)
]

# Display the top 3 locations with the highest price variation
top_3_volatile_locations = filtered_price_std.sort_values(ascending=False).head(3)

top_3_volatile_locations

location
Bommenahalli             21513.992930
Banashankari Stage II     9621.898792
1st Block Koramangala     9617.805384
Name: price_per_sqft, dtype: float64

Question 8 — Micro-Market Dominance
Assignment requirement: For each of the three busiest locations (the locations with the most listings), identify the top two societies with the highest number of listings.

In [29]:
# Find the three locations with the most property listings
top_3_busiest_locations = df["location"].value_counts().head(3)
top_3_busiest_locations

location
Whitefield         531
Sarjapur  Road     388
Electronic City    293
Name: count, dtype: int64

In [30]:
# Keep properties from the three busiest locations
busy_location_df = df[df["location"].isin(top_3_busiest_locations.index)]

# Count listings for each society within each location
society_counts = (
    busy_location_df
    .groupby(["location", "society"])
    .size()
    .reset_index(name="listing_count")
)

# Sort by location and listing count, highest first
society_counts = society_counts.sort_values(
    ["location", "listing_count"],
    ascending=[True, False]
)

# Select the top 2 societies within each location
top_2_societies = society_counts.groupby("location").head(2)

top_2_societies

,location,society,listing_count
11,Electronic City,GMown E,54
31,Electronic City,Prarkun,49
54,Sarjapur Road,Adeatlm,37
108,Sarjapur Road,PualeSk,21
209,Whitefield,Prtates,56
196,Whitefield,Oreldhi,17


Question 9 — Intra-Location Dense Ranking
Assignment requirement: Within each location, rank properties from most expensive to least expensive using a dense rank based on price_per_sqft.

In [31]:
# Rank properties by price per square foot within each location
df["price_rank"] = df.groupby("location")["price_per_sqft"].rank(
    method="dense",
    ascending=False
)

# Display the location, price, and rank for the first 10 properties
df[["location", "price_per_sqft", "price_rank"]].head(10)

,location,price_per_sqft,price_rank
0,Electronic City Phase II,3699.810606,46.0
1,Chikka Tirupathi,4615.384615,6.0
2,Uttarahalli,4305.555556,46.0
3,Lingadheeranahalli,6245.890861,17.0
4,Kothanur,4250.000000,53.0
5,Whitefield,3247.863248,460.0
6,Old Airport Road,7467.057101,12.0
7,Rajaji Nagar,18181.818182,9.0
8,Marathahalli,4828.244275,128.0
10,Whitefield,3888.888889,417.0


Question 10 — Locality Average Delta
Assignment requirement: Use groupby.transform() to add a column named location_avg_price, then calculate the exact percentage deviation of each property's price relative to its locality's average.

In [32]:
# Calculate the average property price within each location
df["location_avg_price"] = df.groupby("location")["price"].transform("mean")

# Calculate percentage deviation from the location average price
df["price_deviation_pct"] = (
    (df["price"] - df["location_avg_price"])
    / df["location_avg_price"]
) * 100

# Display prices, locality averages, and percentage deviations
df[["location", "price", "location_avg_price", "price_deviation_pct"]].head(10)

,location,price,location_avg_price,price_deviation_pct
0,Electronic City Phase II,39.07,48.026190,-18.648555
1,Chikka Tirupathi,120.00,115.015882,4.333417
2,Uttarahalli,62.00,56.266500,10.189900
3,Lingadheeranahalli,95.00,115.348261,-17.640718
4,Kothanur,51.00,95.025397,-46.330137
5,Whitefield,38.00,127.431864,-70.180143
6,Old Airport Road,204.00,184.080303,10.821200
7,Rajaji Nagar,600.00,323.673333,85.372083
8,Marathahalli,63.25,101.920000,-37.941523
10,Whitefield,70.00,127.431864,-45.068684


Question 11 — Create four price tiers
Concept first: pd.qcut() divides numerical data into quantile-based groups. Here, we'll divide price_per_sqft into four tiers: Budget, Standard, Premium, and Luxury.

In [33]:
# Divide properties into four price-per-square-foot tiers
df["price_tier"] = pd.qcut(
    df["price_per_sqft"],
    q=4,
    labels=["Budget", "Standard", "Premium", "Luxury"]
)

# Display the first 10 properties with their price tiers
df[["location", "price_per_sqft", "price_tier"]].head(10)

,location,price_per_sqft,price_tier
0,Electronic City Phase II,3699.810606,Budget
1,Chikka Tirupathi,4615.384615,Standard
2,Uttarahalli,4305.555556,Standard
3,Lingadheeranahalli,6245.890861,Premium
4,Kothanur,4250.000000,Standard
5,Whitefield,3247.863248,Budget
6,Old Airport Road,7467.057101,Luxury
7,Rajaji Nagar,18181.818182,Luxury
8,Marathahalli,4828.244275,Standard
10,Whitefield,3888.888889,Budget


Question 12 — Rolling average for Whitefield
Concept first: A rolling average calculates the average over a moving window of rows. Here, each window contains 3 properties, after sorting Whitefield properties by total_sqft.

In [34]:
# Select properties located in Whitefield
whitefield_df = df[df["location"] == "Whitefield"].copy()

# Sort properties by total square feet
whitefield_df = whitefield_df.sort_values("total_sqft")

# Calculate the rolling average of price over 3 properties
whitefield_df["rolling_avg_price"] = (
    whitefield_df["price"].rolling(window=3).mean()
)

# Display the property size, price, and rolling average
whitefield_df[["total_sqft", "price", "rolling_avg_price"]].head(10)

,total_sqft,price,rolling_avg_price
10752,516.0,39.00,NaN
10559,524.0,29.00,NaN
5782,530.0,29.44,32.480000
7036,605.0,40.00,32.813333
13212,613.0,48.00,39.146667
4541,624.0,35.00,41.000000
5848,630.5,32.15,38.383333
3364,630.5,32.79,33.313333
7843,640.0,19.83,28.256667
5044,650.0,25.00,25.873333


Question 13 — Cumulative property value in Sarjapur Road
Concept first: A cumulative sum keeps adding each property's price to the total of all preceding rows. We'll sort by bhk in ascending order first.

In [35]:
# Select properties in Sarjapur Road
sarjapur_df = df[df["location"] == "Sarjapur  Road"].copy()

# Sort properties by number of bedrooms in ascending order
sarjapur_df = sarjapur_df.sort_values("bhk")

# Calculate cumulative property value
sarjapur_df["cumulative_price"] = sarjapur_df["price"].cumsum()

# Display BHK, price, and cumulative price
sarjapur_df[["bhk", "price", "cumulative_price"]].head(10)

,bhk,price,cumulative_price
1180,1,27.00,27.00
12324,1,26.00,53.00
11709,1,35.80,88.80
10704,1,27.00,115.80
10319,1,31.10,146.90
9364,1,19.65,166.55
9603,1,52.00,218.55
6655,1,45.00,263.55
6291,1,39.90,303.45
4758,1,29.00,332.45


Question 14 — Price difference from the next-largest property
Concept first: shift(-1) brings the next row's value into the current row. We'll sort properties by total_sqft in descending order within each location, then compare each property's price with the next smaller property in that ordering.

In [36]:
# Sort properties by location and size, largest first
df_sorted = df.sort_values(
    ["location", "total_sqft"],
    ascending=[True, False]
).copy()

# Get the next property's size and price within each location
df_sorted["next_total_sqft"] = (
    df_sorted.groupby("location")["total_sqft"].shift(-1)
)
df_sorted["next_price"] = (
    df_sorted.groupby("location")["price"].shift(-1)
)

# Calculate the price difference from the next property
df_sorted["price_difference"] = (
    df_sorted["price"] - df_sorted["next_price"]
)

# Display the results for the first 10 rows
df_sorted[
    ["location", "total_sqft", "price", "next_total_sqft",
     "next_price", "price_difference"]
].head(10)

,location,total_sqft,price,next_total_sqft,next_price,price_difference
2557,Anekal,351.0,16.0,NaN,NaN,NaN
8385,Banaswadi,527.0,35.0,NaN,NaN,NaN
7638,Basavangudi,670.0,50.0,NaN,NaN,NaN
3620,Devarabeesana Halli,1750.0,149.0,1750.0,150.0,-1.0
6120,Devarabeesana Halli,1750.0,150.0,1672.0,150.0,0.0
3281,Devarabeesana Halli,1672.0,150.0,1670.0,160.0,-10.0
6000,Devarabeesana Halli,1670.0,160.0,1296.0,70.0,90.0
10745,Devarabeesana Halli,1296.0,70.0,1100.0,70.0,0.0
2764,Devarabeesana Halli,1100.0,70.0,NaN,NaN,NaN
8954,Devarachikkanahalli,1705.0,75.0,1700.0,71.0,4.0


Question 15 — Locations where 2 BHK costs more than 3 BHK
Concept first: We'll group properties by location and bhk, calculate the average price for each group, and compare the 2 BHK average with the 3 BHK average in each location.

In [37]:
# Calculate average property price by location and BHK
location_bhk_avg = (
    df.groupby(["location", "bhk"])["price"]
    .mean()
    .unstack()
)

# Keep locations where the average 2 BHK price exceeds the average 3 BHK price
locations_2bhk_higher = location_bhk_avg[
    (location_bhk_avg[2] > location_bhk_avg[3])
    & location_bhk_avg[2].notna()
    & location_bhk_avg[3].notna()
]

# Display the matching locations and average prices
locations_2bhk_higher[[2, 3]]

bhk,2,3
location,,
Abbigere,38.670769,38.266667
Akshaya Vana,53.500000,49.000000
Akshya Nagar,66.423333,58.000000
BEL Road,156.500000,150.000000
BTM 4th Stage,90.000000,55.000000
Babusapalaya,51.834286,50.683000
Banashankari Stage III,84.280000,79.289091
Basava Nagar,70.000000,60.100000
Basavanapura,57.886000,46.740000


Question 16 — Crosstab of BHK distribution by area type
Concept first: pd.crosstab() creates a frequency table showing how two categorical variables relate. Here, we'll show the percentage distribution of BHK configurations within each area_type.

In [38]:
# Create a percentage distribution table of BHK by area type
bhk_area_percentage = pd.crosstab(
    df["area_type"],
    df["bhk"],
    normalize="index"
) * 100

# Display the percentage table
bhk_area_percentage

bhk,1,2,3,4,5,6,7,8,9,10,11,13,16
area_type,,,,,,,,,,,,,
Built-up Area,6.494624,42.967742,32.817204,12.000000,2.580645,1.634409,0.817204,0.387097,0.258065,0.043011,0.000000,0.000000,0.000000
Carpet Area,12.195122,39.024390,36.585366,8.536585,1.219512,0.000000,0.000000,1.219512,1.219512,0.000000,0.000000,0.000000,0.000000
Plot Area,4.692082,19.721408,26.832845,35.997067,7.478006,2.932551,0.953079,0.953079,0.293255,0.146628,0.000000,0.000000,0.000000
Super built-up Area,4.725135,48.127233,41.085629,5.266797,0.414890,0.138297,0.115247,0.023049,0.034574,0.023049,0.023049,0.011525,0.011525


Question 17 — Identify the highest-paid property's society in each location
Concept first: We'll find the property with the highest price in each location, retrieve its society, and use transform() to attach that society name to every row in the same location—without writing a loop.

In [39]:
# Find the society of the highest-priced property in each location
highest_price_idx = df.groupby("location")["price"].idxmax()

# Map each location to the society of its highest-priced property
highest_paid_society = df.loc[highest_price_idx].set_index("location")["society"]

# Add the highest-paid property's society to every row
df["top_society_in_location"] = df["location"].map(highest_paid_society)

# Display the results
df[["location", "price", "society", "top_society_in_location"]].head(10)

,location,price,society,top_society_in_location
0,Electronic City Phase II,39.07,Coomee,NaN
1,Chikka Tirupathi,120.00,Theanmp,Theanmp
2,Uttarahalli,62.00,NaN,NaN
3,Lingadheeranahalli,95.00,Soiewre,Soiewre
4,Kothanur,51.00,NaN,NaN
5,Whitefield,38.00,DuenaTa,Adowslm
6,Old Airport Road,204.00,Jaades,NaN
7,Rajaji Nagar,600.00,Brway G,NaN
8,Marathahalli,63.25,NaN,NaN
10,Whitefield,70.00,NaN,Adowslm


Question 18 — Most and least available property types by location
Concept first: The availability column contains text metadata, such as dates or status labels. We'll count the availability values within each location and identify the least frequent and most frequent values.

In [40]:
# Count each availability value within every location
availability_counts = (
    df.groupby(["location", "availability"])
    .size()
    .reset_index(name="listing_count")
)

# Find the least frequent availability value in each location
least_available = availability_counts.loc[
    availability_counts.groupby("location")["listing_count"].idxmin()
].rename(columns={
    "availability": "least_available_type",
    "listing_count": "least_count"
})

# Find the most frequent availability value in each location
most_available = availability_counts.loc[
    availability_counts.groupby("location")["listing_count"].idxmax()
].rename(columns={
    "availability": "most_available_type",
    "listing_count": "most_count"
})

# Combine both results into one table
availability_summary = least_available[
    ["location", "least_available_type", "least_count"]
].merge(
    most_available[["location", "most_available_type", "most_count"]],
    on="location"
)

# Display the first 10 locations
availability_summary.head(10)

,location,least_available_type,least_count,most_available_type,most_count
0,Anekal,Ready To Move,1,Ready To Move,1
1,Banaswadi,Ready To Move,1,Ready To Move,1
2,Basavangudi,Ready To Move,1,Ready To Move,1
3,Devarabeesana Halli,Ready To Move,6,Ready To Move,6
4,Devarachikkanahalli,18-May,1,Ready To Move,13
5,Electronic City,Ready To Move,2,Ready To Move,2
6,Mysore Highway,Ready To Move,4,Ready To Move,4
7,Rachenahalli,18-May,1,18-May,1
8,Sector 1 HSR Layout,Ready To Move,1,Ready To Move,1
9,Thanisandra,18-Nov,1,18-Nov,1


Question 19 — Calculate missing-value percentages and drop columns above 35%
Concept first: We'll calculate the percentage of missing values in each column. Any column with more than 35% missing values will be dropped from the cleaned DataFrame.

In [41]:
# Calculate the percentage of missing values in each column
missing_percentage = (df.isna().mean() * 100).sort_values(ascending=False)

# Display missing-value percentages
print(missing_percentage)

# Identify columns with more than 35% missing values
columns_to_drop = missing_percentage[missing_percentage > 35].index

# Drop those columns from the dataset
df = df.drop(columns=columns_to_drop)

# Display the remaining columns and dataset shape
print("Dropped columns:", list(columns_to_drop))
print("Remaining shape:", df.shape)

top_society_in_location    54.643316
society                    38.279242
balcony                     3.526671
location                    0.008033
price_deviation_pct         0.008033
location_avg_price          0.008033
price_rank                  0.008033
area_type                   0.000000
bath                        0.000000
total_sqft                  0.000000
availability                0.000000
size                        0.000000
price_per_sqft              0.000000
bhk                         0.000000
price                       0.000000
price_tier                  0.000000
dtype: float64
Dropped columns: ['top_society_in_location', 'society']
Remaining shape: (12448, 14)


Question 20 — Create the master summary by location
We'll create one summary row per location containing all five required metrics:
1. Total Listings — number of properties.
2. Total Revenue Capacity — sum of property prices.
3. Mean BHK — average number of bedrooms.
4. Median Price per Sqft — median price_per_sqft.
5. Dominant Area Type — most frequent area_type.

In [42]:
# Find the most common area type in each location
dominant_area_type = (
    df.groupby("location")["area_type"]
    .agg(lambda x: x.mode().iloc[0])
)

# Create the master summary table
master_summary = df.groupby("location").agg(
    Total_Listings=("price", "count"),
    Total_Revenue_Capacity=("price", "sum"),
    Mean_BHK=("bhk", "mean"),
    Median_Price_Per_Sqft=("price_per_sqft", "median")
)

# Add the dominant area type for each location
master_summary["Dominant_Area_Type"] = dominant_area_type

# Display the first 10 locations
master_summary.head(10)

,Total_Listings,Total_Revenue_Capacity,Mean_BHK,Median_Price_Per_Sqft,Dominant_Area_Type
location,,,,,
Anekal,1,16.0,1.0,4558.404558,Built-up Area
Banaswadi,1,35.0,1.0,6641.366224,Super built-up Area
Basavangudi,1,50.0,1.0,7462.686567,Super built-up Area
Devarabeesana Halli,6,749.0,2.666667,8542.857143,Super built-up Area
Devarachikkanahalli,14,878.0,2.428571,4305.148257,Super built-up Area
Electronic City,2,46.5,1.0,4695.924765,Built-up Area
Mysore Highway,4,147.5,1.5,4235.221675,Super built-up Area
Rachenahalli,2,47.8,1.0,5581.818182,Super built-up Area
Sector 1 HSR Layout,1,276.0,1.0,4600.000000,Super built-up Area
